# Differential Equations — Session 19
## Section 4.7: Cauchy–Euler Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Recognize Cauchy–Euler structure.
2. derive the indicial equation from $y=x^m$.
3. handle distinct, repeated, and complex roots.
4. interpret logarithmic oscillation.
5. transform with $t=\ln x$ to constant coefficients.
6. state interval restrictions for $x>0$ and $x<0$.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | Recognition and power trial |\n| 18–42 min | Three root cases |\n| 42–62 min | Logarithmic oscillations |\n| 62–78 min | Transformation $t=\ln x$ |\n| 78–88 min | Numerical/symbolic verification |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 4.7-A — Cauchy–Euler equation

A second-order Cauchy–Euler equation is

$$
ax^2y''+bxy'+cy=g(x).
$$

For the homogeneous equation, $y=x^m$ gives

$$
am(m-1)+bm+c=0.
$$

### Theorem 4.7-B — Root forms

- Distinct real roots $m_1,m_2$: $c_1x^{m_1}+c_2x^{m_2}$.
- Repeated root $m$: $x^m(c_1+c_2\ln x)$.
- Complex roots $\alpha\pm i\beta$: $x^\alpha[c_1\cos(\beta\ln x)+c_2\sin(\beta\ln x)]$.

These forms are naturally stated on $x>0$. Separate treatment is needed on $x<0$.

### Proposition 4.7-C — Logarithmic transformation

With $t=\ln x$, a Cauchy–Euler equation becomes a constant-coefficient equation in $t$.

### Classroom Checkpoint — Constant-Coefficient Transformation

Which change of independent variable converts a Cauchy–Euler equation on $x>0$ into a constant-coefficient equation?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Three cases

In [ ]:
def euler_case(case='distinct',c1=1.0,c2=1.0):
    x=np.logspace(-2,2,1000)
    if case=='distinct': y=c1*x**2+c2*x**-3; title='roots 2 and -3'
    elif case=='repeated': y=x**2*(c1+c2*np.log(x)); title='repeated root 2'
    else: y=x**0.5*(c1*np.cos(2*np.log(x))+c2*np.sin(2*np.log(x))); title='roots 0.5 ± 2i'
    plt.semilogx(x,y); plt.xlabel('x (log scale)'); plt.ylabel('y'); plt.title(title); plt.show()
if WIDGETS_AVAILABLE: interact(euler_case,case=Dropdown(options=['distinct','repeated','complex'],value='distinct'),c1=FloatSlider(min=-3,max=3,step=.25,value=1),c2=FloatSlider(min=-3,max=3,step=.25,value=1))
else: euler_case()

## 2. Log-periodic oscillation

For complex roots, zeros are equally spaced in $\ln x$, not in $x$. The oscillations stretch geometrically along the positive axis.

In [ ]:
x=np.logspace(-3,3,3000); y=np.cos(2*np.log(x));
plt.semilogx(x,y); plt.title(r'$\cos(2\ln x)$ is periodic in $\ln x$'); plt.show()

## 3. Transforming to constant coefficients

Let $Y(t)=y(e^t)$. Then

$$xy'=Y_t,\qquad x^2y''=Y_{tt}-Y_t.$$

Thus

$$ax^2y''+bxy'+cy=aY_{tt}+(b-a)Y_t+cY.$$

In [ ]:
t=sp.symbols('t',real=True); Y=sp.Function('Y'); a,b,c=sp.symbols('a b c')
print('Transformed operator: a Y_tt + (b-a) Y_t + c Y')

## 4. Example

For

$$x^2y''-3xy'+4y=0,$$

the indicial equation is $(m-2)^2=0$, giving

$$y=x^2(c_1+c_2\ln x).$$

In [ ]:
x=sp.symbols('x',positive=True); c1,c2=sp.symbols('c1 c2'); y=x**2*(c1+c2*sp.log(x)); display(sp.simplify(x**2*sp.diff(y,x,2)-3*x*sp.diff(y,x)+4*y))

## Classroom Checkpoint — Exit Check

Solve $x^2y''+xy'+9y=0$ on $x>0$.

> Pause here. Let students commit to an answer before running the next cell.